In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [4]:
params = {
   'axes.labelsize': 21,
   'font.size': 16,
   'font.family': 'sans-serif',
   'font.serif': 'Arial',
   'legend.fontsize': 18,
   'xtick.labelsize': 18,
   'ytick.labelsize': 18,
   'axes.labelpad': 15,

   'figure.figsize': [10,8], # value in inches based on dpi of monitor
   'figure.dpi': 105.5, # My monitor has a dpi of around 105.5 px/inch

   'axes.grid': True,
   'grid.linestyle': '-',
   'grid.alpha': 0.25,
   'axes.linewidth': 1,
   'figure.constrained_layout.use': True,


   # Using Paul Tol's notes:
   'axes.prop_cycle':
      mpl.cycler(color=['#4477aa', # blue
                        '#ee6677', # red/pink
                        '#228833', # green
                        '#aa3377', # purple
                        '#66ccee', # cyan
                        '#ccbb44', # yellow
                        '#bbbbbb', # grey
                        ]),

      # Pick either the cycler above, or the cycler below:

      # (mpl.cycler(color=['#4477aa', # blue
      #                     '#ee6677', # red/pink
      #                     '#228833', # green
      #                     '#aa3377', # purple
      #                     '#66ccee', # cyan
      #                     '#ccbb44', # yellow
      #                     '#bbbbbb', # grey
      #                     ]) +
      #   mpl.cycler(linestyle=['-', # solid
      #                         '--', # dashed
      #                         ':', # dotted
      #                         '-.', # dash dot
      #                         (0, (3, 1, 1, 1, 1, 1)), # narrow dash dot dot
      #                         (0, (1, 2, 5, 2, 5, 2)), # dash dash dot
      #                         (0, (5, 2.5, 1, 2.5, 1, 2.5)), # dash dot dot
      #                         ])),

   'lines.linewidth': 2.5,

   'image.cmap': 'jet',
}


plt.rcParams.update(params)

In [22]:
voltages = [550,560,570,580,590,600,610]
charge_data_0 = np.zeros_like(voltages,dtype=float)
light_data_0 = np.zeros_like(voltages,dtype=float)
charge_err_0 = np.zeros_like(voltages,dtype=float)
light_err_0 = np.zeros_like(voltages,dtype=float)
tracked_levels = ['11.50','11.63','11.88','12.13','12.38','12.50','12.63','12.88','13.13','13.38','13.63','13.88','14.00','14.13','14.38','14.63','14.88','15.13','15.38','15.63','15.88','16.13','16.38','16.63','16.88','17.13','17.38','17.63','17.88','18.13','18.38','18.63','18.88','19.13','19.38','19.63']
p_per_e_0 = np.zeros_like(voltages,dtype=float)
p_per_e_err_0 = np.zeros_like(voltages,dtype=float)

In [23]:
min_id = 313.9
max_id = 319.7

In [28]:
for voltage in voltages:
    gain_list = []
    num = 0
    sizes = []
    all=[]

    # Light dicts stores all colls dicts
    light_dicts = []

    for i in range(1,5001,1):

        # colls_dict stores level ID and number of collisions within it for each run
        colls_dict = {}

        # read in
        try:
            infile = open("/Users/tomszwarcer/Documents/MIGDAL/UPDATE/light_output/charge_light_output/cl_"+str(voltage)+"_0/" + str(i) + ".csv","r")
        except FileNotFoundError:
            continue
        all = infile.readlines()
        gain_list.append(int(all[0]))

        # process each level
        for row in range(1,len(all)):
            ncoll = all[row].split(",")[1]
            id = format(round(float(all[row].split(",")[0]),2),'.2f')
            id = "0" + str(id)
            if min_id <= float(id) <= max_id:
                colls_dict[id]=int(ncoll)
        num += 1

        light_dicts.append(colls_dict)
    print(str(num) + " data points for "+str(voltage)+"V")

    light_per_run = []
    sum = 0
    for i in light_dicts:
        for j in i:
            sum+=i[j]
        light_per_run.append(sum)
        sum=0

    #### GAIN

    n_bins = 35

    #calculate mean
    total_gain = 0
    num_runs = len(gain_list)
    for gain in gain_list:
        total_gain += gain
    mean = total_gain/num_runs

    #calculate variance
    var = 0
    for gain in gain_list:
        var += (gain - mean)**2
    var = var/num_runs
    se = np.sqrt(var)/np.sqrt(num_runs)

    charge_data_0[voltages.index(voltage)] = mean
    charge_err_0[voltages.index(voltage)] = se

    #### LIGHT

    n_bins = 35

    #calculate mean
    total_light = 0
    num_runs = len(light_per_run)
    for data in light_per_run:
        total_light += data
    mean = total_light/num_runs

    #calculate variance
    var = 0
    for data in light_per_run:
        var += (data - mean)**2
    var = var/num_runs
    se = np.sqrt(var)/np.sqrt(num_runs)

    light_data_0[voltages.index(voltage)] = mean
    light_err_0[voltages.index(voltage)] = se

    plt.hist(light_per_run,bins=n_bins,density=False)
    plt.xlabel("Light production (Arb. units)")
    plt.ylabel("Counts")
    plt.title("Light distribution [Pure CF4, 60 Torr, dV = "+str(voltage)+"]")
    plt.text(x=0.5*plt.xlim()[1],y=0.75*plt.ylim()[1],s="mean = " + str(int(round(mean,0))) + "\nSD = " + str(int(round(se,0))))
    plt.savefig("plots/light_"+str(voltage)+"_0.png")
    plt.close()



1969 data points for 550V
1963 data points for 560V
1890 data points for 570V
1828 data points for 580V
1964 data points for 590V
1978 data points for 600V
1968 data points for 610V


In [ ]:
#fraction of collisions that result in scintillation
scintillation_fraction = 0.12

#efficiency of transfer out of GEM system
#assumes zero error in this value, needs further investigation
efficiency = 0.17585

#discrepancy factor of gas gain
gain_discrepancy = 2

#light correction
light_data_0 = light_data_0*scintillation_fraction
light_err_0 = light_err_0*scintillation_fraction

#charge correction: gain is off by ~2
charge_data_0 = np.multiply(charge_data_0,gain_discrepancy)
charge_err_0 = np.multiply(charge_err_0,gain_discrepancy)

frac_err_charge = np.divide(charge_err_0,charge_data_0)
frac_err_light = np.divide(light_err_0,light_data_0)
quadrature = np.add(np.square(frac_err_charge),np.square(frac_err_light))
p_per_e_err_0 = np.sqrt(quadrature)

p_per_e_0 = np.multiply(np.divide(light_data_0,charge_data_0),efficiency)
p_per_e_err_0 = np.multiply(p_per_e_0,p_per_e_err_0)




In [26]:
def linear(a,x):
    return a*x

In [35]:
ax = plt.axes()

# 0%
ax.scatter(charge_data_0,light_data_0,label="Simulation data")
ax.errorbar(charge_data_0,light_data_0,xerr = charge_err_0,yerr = light_err_0,ls='none')

ax.set_xlabel("Electrons collected")
ax.set_ylabel("Photons produced")
ax.set_title("[60 Torr, pure CF4, scint. probability = "+str(scintillation_fraction)+"]")

parameters, cov_matrix = curve_fit(linear, charge_data_0, light_data_0, p0=[0.3])
print(parameters)
x = np.array([0,max(charge_data_0)])
y=parameters[0]*x
ax.plot(x,y,label='Fitted line (m = '+str(round(parameters[0],2))+')')
ax.legend()
plt.savefig("plots/p_per_e.png")
plt.close()


[0.3564948]
